# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print the name and description
print(f"{getattr(metadata, 'name')}: {getattr(metadata, 'description')}")

## 2. Data Overview
Review the available record sets (tables), fields, and their IDs.

In [ ]:
# List all record sets and their field IDs
rs_list = list(dataset.record_sets)
print(f"Record sets found ({len(rs_list)}):")
for i, record_set in enumerate(rs_list):
    print(f"  {i+1}. Name: {record_set.name}, @id: {record_set.id}")
    field_ids = [field.id for field in record_set.fields]
    print(f"       Field @ids: {field_ids}\n")
# As an example, let's print the first 2 records from each record set
for record_set in rs_list:
    records_iter = dataset.records(record_set=record_set.id)
    print(f"Sample records from record set {record_set.id}:")
    for i, rec in enumerate(records_iter):
        if i > 1:
            break
        print(rec)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data for all available record sets by @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {df.shape[0]} rows with columns: {df.columns.tolist()} from record set {rs_id}\n")

# For demonstration, focus on the main survey record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    print(f"Columns in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter, normalize, group. We use the field `@id`s for all operations.

In [ ]:
# Example: Analyze GAD-7 total score (anxiety assessment)
# You may need to adjust the IDs to match your dataset, e.g. gad7_total_score, phq9_total_score, psq_total_score, etc.
survey_df = dataframes.get(main_record_set_id)

# Suppose the GAD-7 total score column is '@gad7_total_score' (replace with correct field @id!)
numeric_field = None

# Try to auto-detect likely GAD-7/PHQ-9/PSQ total score fields by scanning column names
for col in survey_df.columns:
    if 'gad7' in col.lower() and 'score' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    for col in survey_df.columns:
        if 'phq' in col.lower() and 'score' in col.lower():
            numeric_field = col
            break
if numeric_field is None:
    for col in survey_df.columns:
        if 'psq' in col.lower() and 'score' in col.lower():
            numeric_field = col
            break

if numeric_field:
    print(f"Analyzing numeric field '{numeric_field}'")
    # Drop missing entries
    filtered_df = survey_df[survey_df[numeric_field].notnull()].copy()
    threshold = 10
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} ({len(filtered_df)} records):")
    display(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by another field, e.g. 'village' or 'gender'
    group_field = None
    for g in ['village', 'gender', 'level_of_education', 'age_group']:
        for col in survey_df.columns:
            if g in col.lower():
                group_field = col
                break
        if group_field:
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by '{group_field}', mean values:")
        display(grouped_df)
    else:
        print("No suitable group field found for grouping analysis.")
else:
    print("No GAD-7, PHQ-9, or PSQ total score field found in columns for EDA.")

## 5. Visualization
Visualize data distributions or field relationships in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id and numeric_field:
    # Histogram of the selected score
    plt.figure(figsize=(8,5))
    sns.histplot(survey_df[numeric_field].dropna().astype(float), bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # Boxplot of the numeric field by group (e.g. gender), if available
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=survey_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load a FAIR² Croissant dataset schema, review its structure, extract records using logical `@id` references, and perform simple exploratory analysis and visualization. 

- All table, field, and column references are handled via their Croissant-assigned `@id`.
- The dataset contains rich demographic and mental health indicators (such as GAD-7, PHQ-9, PSQ scores) for Kilifi County, Kenya.
- You can extend this notebook for advanced statistical and ML analysis, ensuring you always reference data components by their `@id`s.

_For more details, visit the original [FAIR² Mental Health Survey Dataset Croissant source](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)._